In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install torchmetrics

In [ ]:
!pip install pytorch_lightning

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import random
import os
import json
from transformers import AdamW, get_linear_schedule_with_warmup
from torchmetrics.functional import auroc
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
import pytorch_lightning as pl
from torchmetrics import AUROC, Accuracy, F1Score
from tqdm.auto import tqdm
from sklearn.metrics import classification_report

In [ ]:
csv_path1 = '/content/drive/MyDrive/goemotions1.csv'
csv_path2 = '/content/drive/MyDrive/goemotions2.csv'

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, data: pd.DataFrame, tokenizer: BertTokenizer, label_names, max_token_len: int = 128):
        self.tokenizer = tokenizer
        self.data = data
        self.max_token_len = max_token_len
        self.LABEL_COLUMNS = label_names

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index: int):
        data_row = self.data.iloc[index]
        comment_text = data_row.text
        labels = data_row[self.LABEL_COLUMNS]
        encoding = self.tokenizer.encode_plus(
          comment_text,
          add_special_tokens=True,
          max_length=self.max_token_len,
          return_token_type_ids=False,
          padding="max_length",
          truncation=True,
          return_attention_mask=True,
          return_tensors='pt',
        )
        return dict(
          comment_text=comment_text,
          input_ids=encoding["input_ids"].flatten(),
          attention_mask=encoding["attention_mask"].flatten(),
          labels=torch.FloatTensor(labels.values.astype(np.float32))
        )


In [ ]:
class CustomDataModule(pl.LightningDataModule):
    def __init__(self, train_df, test_df, tokenizer, label_names, batch_size=8, max_token_len=128):
        super().__init__()
        self.batch_size = batch_size
        self.train_df = train_df
        self.test_df = test_df
        self.tokenizer = tokenizer
        self.max_token_len = max_token_len
        self.label_names = label_names

    def setup(self, stage=None):
        self.train_dataset = CustomDataset(
            self.train_df,
            self.tokenizer,
            self.label_names,
            self.max_token_len
        )
        self.test_dataset = CustomDataset(
            self.test_df,
            self.tokenizer,
            self.label_names,
            self.max_token_len
        )

    def train_dataloader(self):
        return DataLoader(
          self.train_dataset,
          batch_size=self.batch_size,
          shuffle=True,
          num_workers=2
        )

    def val_dataloader(self):
        return DataLoader(
          self.test_dataset,
          batch_size=self.batch_size,
          num_workers=2
        )

    def test_dataloader(self):
        return DataLoader(
          self.test_dataset,
          batch_size=self.batch_size,
          num_workers=2
        )

In [ ]:
class CustomBertModel(pl.LightningModule):
    def __init__(self, n_classes: int, labels, n_training_steps=None, n_warmup_steps=None):
        super().__init__()
        self.bert = BertModel.from_pretrained(BERT_MODEL_NAME, return_dict=True)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_classes)
        self.n_training_steps = n_training_steps
        self.n_warmup_steps = n_warmup_steps
        self.criterion = nn.BCELoss()
        self.LABEL_COLUMNS = labels

        # Initialize F1 score metric for multi-label classification
        self.train_f1 = F1Score(task="multilabel", num_labels=n_classes, threshold=0.5)
        self.val_f1 = F1Score(task="multilabel", num_labels=n_classes, threshold=0.5)

    def forward(self, input_ids, attention_mask, labels=None):
        output = self.bert(input_ids, attention_mask=attention_mask)
        output = self.classifier(output.pooler_output)
        output = torch.sigmoid(output)
        loss = 0
        if labels is not None:
            loss = self.criterion(output, labels)
        return loss, output

    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        loss, outputs = self(input_ids, attention_mask, labels)
        self.log("train_loss", loss, prog_bar=True, logger=True, on_step=True, on_epoch=True)

        # Update F1 score
        self.train_f1(outputs, labels)

        # Log F1 score on each step
        self.log("train_f1", self.train_f1, prog_bar=True, logger=True, on_step=True, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        loss, outputs = self(input_ids, attention_mask, labels)
        self.log("val_loss", loss, prog_bar=True, logger=True, on_step=True, on_epoch=True)

        # Update F1 score
        self.val_f1(outputs, labels)

        # Log F1 score on each step
        self.log("val_f1", self.val_f1, prog_bar=True, logger=True, on_step=True, on_epoch=True)

        return loss

    def on_train_epoch_end(self):
        # Reset F1 score at the end of the epoch
        self.train_f1.reset()

    def on_validation_epoch_end(self):
        # Reset F1 score at the end of the epoch
        self.val_f1.reset()

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=2e-5)
        scheduler = get_linear_schedule_with_warmup(
          optimizer,
          num_warmup_steps=self.n_warmup_steps,
          num_training_steps=self.n_training_steps
        )
        return dict(
          optimizer=optimizer,
          lr_scheduler=dict(
            scheduler=scheduler,
            interval='step'
          )
        )

In [ ]:
TRAINED_MODEL_PATH_EMO_1 = '/content/drive/MyDrive/QTag-epoch=14-val_loss=0.08_without_preprocessing.ckpt';
TRAINED_MODEL_PATH_EMO_2 = '/content/drive/MyDrive/QTag-epoch=14-val_loss=0.08_emotions2.ckpt';
TRAINED_MODEL_PATH_EMO_3 = '/content/drive/MyDrive/QTag-epoch=14-val_loss=0.08_emotions3.ckpt';



In [ ]:
df_train_emo1 = pd.read_csv(r"/content/drive/MyDrive/goemotions1.csv")
df_train_emo2 = pd.read_csv(r"/content/drive/MyDrive/goemotions2.csv")
df_train_emo3 = pd.read_csv(r"/content/drive/MyDrive/goemotions3.csv")

label_list_emo1 = list(df_train_emo1.columns[9:])
label_list_emo2 = list(df_train_emo2.columns[9:])
label_list_emo3 = list(df_train_emo3.columns[9:])

In [ ]:
BERT_MODEL_NAME = 'bert-base-cased'
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)
MAX_TOKEN_COUNT = 128
THRESHOLD = 0.5


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
trained_model_emo_1 = CustomBertModel.load_from_checkpoint(
  TRAINED_MODEL_PATH_EMO_1,
  labels=label_list_emo1,
  n_classes=len(label_list_emo1)
)
trained_model_emo_1.eval()
trained_model_emo_1.freeze()
trained_model_emo_1 = trained_model_emo_1.to(device)



model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [ ]:
trained_model_emo_2 = CustomBertModel.load_from_checkpoint(
  TRAINED_MODEL_PATH_EMO_2,
  labels=label_list_emo2,
  n_classes=len(label_list_emo2)
)
trained_model_emo_2.eval()
trained_model_emo_2.freeze()
trained_model_emo_2 = trained_model_emo_2.to(device)



In [ ]:
trained_model_emo_3 = CustomBertModel.load_from_checkpoint(
  TRAINED_MODEL_PATH_EMO_3,
  labels=label_list_emo3,
  n_classes=len(label_list_emo3)
)
trained_model_emo_3.eval()
trained_model_emo_3.freeze()
trained_model_emo_3 = trained_model_emo_3.to(device)



In [ ]:
df_val_goemotions1 = pd.read_csv(r"/content/drive/MyDrive/goemotions1.csv")

val_dataset_goemotions1 = CustomDataset(
  df_val_goemotions1,
  tokenizer,
  label_list_emo1,
  max_token_len=MAX_TOKEN_COUNT
)

In [ ]:
df_val_goemotions2 = pd.read_csv(r"/content/drive/MyDrive/goemotions2.csv")

val_dataset_goemotions2 = CustomDataset(
  df_val_goemotions2,
  tokenizer,
  label_list_emo2,
  max_token_len=MAX_TOKEN_COUNT
)

In [ ]:
df_val_goemotions3 = pd.read_csv(r"/content/drive/MyDrive/goemotions3.csv")

val_dataset_goemotions3 = CustomDataset(
  df_val_goemotions3,
  tokenizer,
  label_list_emo3,
  max_token_len=MAX_TOKEN_COUNT
)

In [ ]:
trained_models = [trained_model_emo_1, trained_model_emo_2, trained_model_emo_3]
val_datasets = [val_dataset_goemotions1, val_dataset_goemotions2, val_dataset_goemotions3]
label_lists = [label_list_emo1, label_list_emo2, label_list_emo3]

predictions_per_dataset = []
labels_per_dataset = []

for model, dataset, label_list in zip(trained_models, val_datasets, label_lists):
  predictions = []
  labels = []
  for item in tqdm(dataset):
      _, prediction = model(
        item["input_ids"].unsqueeze(dim=0).to(device),
        item["attention_mask"].unsqueeze(dim=0).to(device)
      )
      predictions.append(prediction.flatten())
      labels.append(item["labels"].int())

  predictions = torch.stack(predictions).detach().cpu()
  labels = torch.stack(labels).detach().cpu()

  predictions_per_dataset.append(predictions)
  labels_per_dataset.append(labels)

  0%|          | 0/70000 [00:00<?, ?it/s]

  0%|          | 0/70000 [00:00<?, ?it/s]

  0%|          | 0/71225 [00:00<?, ?it/s]

In [ ]:
for predictions, labels, label_list in zip(predictions_per_dataset, labels_per_dataset, label_lists):
  try:
      print("AUROC per tag")
      for i, name in enumerate(label_list):
          tag_auroc = auroc(predictions[:, i], labels[:, i], 'binary')
          print(f"{name}: {tag_auroc}")
  except Exception as e:
      print('ERROR AUROC: ', e)

  try:
      y_pred = predictions.numpy()
      y_true = labels.numpy()
      upper, lower = 1, 0
      y_pred = np.where(y_pred > THRESHOLD, upper, lower)
      print(classification_report(
        y_true,
        y_pred,
        target_names=label_list,
        zero_division=0
      ))
  except Exception as e:
      print('ERROR classification_report: ', e)

AUROC per tag
admiration: 0.9736242294311523
amusement: 0.973506510257721
anger: 0.9588450789451599
annoyance: 0.9302250146865845
approval: 0.9241296052932739
caring: 0.9215437769889832
confusion: 0.9382791519165039
curiosity: 0.9676618576049805
desire: 0.9110040664672852
disappointment: 0.9048113822937012
disapproval: 0.9360690116882324
disgust: 0.9365396499633789
embarrassment: 0.8780953288078308
excitement: 0.9275215864181519
fear: 0.9412418007850647
gratitude: 0.9860317707061768
grief: 0.7856572866439819
joy: 0.9516129493713379
love: 0.9834840297698975
nervousness: 0.888075590133667
optimism: 0.9291669726371765
pride: 0.7499229907989502
realization: 0.8518598079681396
relief: 0.8027317523956299
remorse: 0.9369245767593384
sadness: 0.9464126229286194
surprise: 0.9305031299591064
neutral: 0.9546303749084473
                precision    recall  f1-score   support

    admiration       0.84      0.69      0.76      5647
     amusement       0.82      0.70      0.75      3081
         a

In [ ]:
class EnsembleBertModel(pl.LightningModule):
    def __init__(self, models, weights=None):
        super().__init__()
        self.models = nn.ModuleList(models)
        if weights is None:
            self.weights = torch.ones(len(models))
        else:
            self.weights = torch.tensor(weights)
        self.weights = nn.Parameter(self.weights / self.weights.sum())

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = []
        for model in self.models:
            _, output = model(input_ids, attention_mask)
            outputs.append(output)

        weighted_outputs = torch.stack([w * o for w, o in zip(self.weights, outputs)])
        ensemble_output = weighted_outputs.sum(dim=0)

        loss = 0
        if labels is not None:
            criterion = nn.BCELoss()
            loss = criterion(ensemble_output, labels)

        return loss, ensemble_output

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        return optimizer

    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        loss, outputs = self(input_ids, attention_mask, labels)
        self.log("train_loss", loss, prog_bar=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        loss, outputs = self(input_ids, attention_mask, labels)
        self.log("val_loss", loss, prog_bar=True, logger=True)
        return loss

In [ ]:
model1 = CustomBertModel.load_from_checkpoint(TRAINED_MODEL_PATH_EMO_1,labels=label_list_emo1,n_classes=len(label_list_emo1))
model2 = CustomBertModel.load_from_checkpoint(TRAINED_MODEL_PATH_EMO_2,labels=label_list_emo2,n_classes=len(label_list_emo2))
model3 = CustomBertModel.load_from_checkpoint(TRAINED_MODEL_PATH_EMO_3,labels=label_list_emo3,n_classes=len(label_list_emo3))

In [ ]:
ensemble_model = EnsembleBertModel([model1, model2, model3], weights=[0.4, 0.3, 0.3])

ensemble_model.eval()
ensemble_model.freeze()

In [ ]:
predictions_per_dataset = []
labels_per_dataset = []

for dataset, label_list in zip(val_datasets, label_lists):
  predictions = []
  labels = []
  for item in tqdm(dataset):
      _, prediction = ensemble_model(
        item["input_ids"].unsqueeze(dim=0).to(device),
        item["attention_mask"].unsqueeze(dim=0).to(device)
      )
      predictions.append(prediction.flatten())
      labels.append(item["labels"].int())

  predictions = torch.stack(predictions).detach().cpu()
  labels = torch.stack(labels).detach().cpu()

  predictions_per_dataset.append(predictions)
  labels_per_dataset.append(labels)

  0%|          | 0/70000 [00:00<?, ?it/s]

  0%|          | 0/70000 [00:00<?, ?it/s]

  0%|          | 0/71225 [00:00<?, ?it/s]

In [ ]:
for predictions, labels, label_list in zip(predictions_per_dataset, labels_per_dataset, label_lists):
  try:
      print("AUROC per tag")
      for i, name in enumerate(label_list):
          tag_auroc = auroc(predictions[:, i], labels[:, i], 'binary')
          print(f"{name}: {tag_auroc}")
  except Exception as e:
      print('ERROR AUROC: ', e)

  try:
      y_pred = predictions.numpy()
      y_true = labels.numpy()
      upper, lower = 1, 0
      y_pred = np.where(y_pred > THRESHOLD, upper, lower)
      print(classification_report(
        y_true,
        y_pred,
        target_names=label_list,
        zero_division=0
      ))
  except Exception as e:
      print('ERROR classification_report: ', e)

AUROC per tag
admiration: 0.9563924074172974
amusement: 0.962003767490387
anger: 0.9422805309295654
annoyance: 0.8929859399795532
approval: 0.8662245273590088
caring: 0.897616982460022
confusion: 0.919913649559021
curiosity: 0.9507680535316467
desire: 0.8943623304367065
disappointment: 0.8807559013366699
disapproval: 0.906425952911377
disgust: 0.923133134841919
embarrassment: 0.8770619034767151
excitement: 0.9073131680488586
fear: 0.9357120990753174
gratitude: 0.9813924431800842
grief: 0.7543245553970337
joy: 0.9309306740760803
love: 0.97742760181427
nervousness: 0.8677027225494385
optimism: 0.9044977426528931
pride: 0.7381902933120728
realization: 0.8017493486404419
relief: 0.7892590761184692
remorse: 0.9285798072814941
sadness: 0.9340513348579407
surprise: 0.9148788452148438
neutral: 0.8864511251449585
                precision    recall  f1-score   support

    admiration       0.77      0.52      0.62      5647
     amusement       0.76      0.56      0.64      3081
         anger 